# Searching and Visualizing Planet Tanager STAC Data

[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/HyperCoast/blob/main/docs/examples/tanager_stac.ipynb)

This notebook demonstrates how to search the [Planet Tanager STAC catalog](https://www.planet.com/data/stac/browser/tanager-core-imagery/catalog.json), visualize the `ortho_visual` Cloud-Optimized GeoTIFF directly, and use the `ortho_radiance_hdf5` asset for spectral signatures.

## Install packages

Uncomment the following line to install the packages.

In [ ]:
# %pip install hypercoast

## Import libraries

In [ ]:
import hypercoast
import geopandas as gpd

## Search Tanager STAC items

The Tanager catalog is organized into thematic collections. The example below searches the Coastal and Water Bodies collection near San Francisco Bay.

## Display Tanager footprints

The [planet-open-data](https://github.com/opengeos/planet-open-data) repository provides GeoJSON files containing the footprints of Tanager products, which are being updated daily. The following code reads the GeoJSON file and visualizes the footprints on a map.

In [ ]:
url = "https://raw.githubusercontent.com/opengeos/planet-open-data/refs/heads/main/data/planet-tanager-sample-products/footprints.geojson"
gdf = gpd.read_file(url)
print(f"Number of footprints: {len(gdf)}")

In [ ]:
gdf.head()

In [ ]:
m = hypercoast.Map()
m.add_gdf(gdf, layer_name="Tanager Footprints")
m

## Search Tanager STAC items

To search Tanager STAC items, draw a rectangle on the map. 

In [ ]:
if m.user_roi_bounds():
    bbox = m.user_roi_bounds()
else:
    bbox = [-123.0, 37.0, -122.0, 38.1]

Search for Tanager data from the [Tanager STAC catalog](https://www.planet.com/data/stac/browser/tanager-core-imagery/catalog.json): 

In [ ]:
items = hypercoast.search_tanager(
    collections="coastal-water-bodies",
    bbox=bbox,
    count=5,
)
len(items), items[0]["id"]

In [ ]:
item = items[0]
item["properties"]["datetime"], item["properties"].get("location_description")

## Visualize the ortho visual COG

Each Tanager STAC item includes an `ortho_visual` asset. It is a Cloud-Optimized GeoTIFF and can be displayed directly with `add_cog_layer()` without downloading the HDF5 spectral product.


In [ ]:
visual_url = hypercoast.get_tanager_asset_url(item, asset="ortho_visual")
visual_url

In [ ]:
m = hypercoast.Map()
m.add_cog_layer(visual_url, name="Tanager ortho visual")
m.zoom = 10
m

## Add a STAC item with spectral signatures

`Map.add_tanager()` accepts a STAC item dictionary or item URL. For STAC inputs, HyperCoast displays the `ortho_visual` COG on the map and downloads `ortho_radiance_hdf5` for the spectral dataset used by the spectral plotting tool.

In [ ]:
m = hypercoast.Map()
m.add_tanager(item, layer_name="Tanager")
m.add("spectral")
m.zoom = 10
m

## Download and read the spectral HDF5 asset

You can also download and read the `ortho_radiance_hdf5` asset explicitly. The reader uses STAC band metadata when the HDF5 file does not contain wavelength metadata.

In [ ]:
dataset = hypercoast.read_tanager_stac(
    item,
    asset="ortho_radiance_hdf5",
    out_dir="data",
)
dataset

In [ ]:
hypercoast.extract_tanager(
    dataset,
    latitude=37.9,
    longitude=-122.3,
    delta=0.01,
    return_plot=True,
)